# Use The Force, Scouts

The Data Bowl dataset grants us a unique view on the movement of players on the field. A player can be tracked by velocity and acceleration in real-time 0.1 second updates.

In the past, a players strength has been assessed by raw ability. A player will have their strength measured through non-football related activities such as a bench press. This player will be measured for height and weight as well for a rough estimate of their strength. The issue with this method is this strength is not reflective of in-game strength.

## Where Force Comes In

The movement of a player can be representative of in-game production. While a bench press measures how strong a player is in a gym setting, there is no guarantee it will translate to an in-game setting. To counteract this, we can calculate a Force statistic using measurements captured during both the combine and actual play and view how they correlate.

## Who Are We Looking At?

Offensive linemen are a position lacking in easily scoutable information. While statistics such as pressures allowed can paint a better picture of an offensive linemen, these statistics can be lacking in information and are heavily dependent on context.

My solution? Force generated during plays. By separating all offensive linemen plays based on run or pass concepts, we can determine the force generated by the player based on their size and acceleration, and determine if this correlates to similar concepts in the NFL Combine.

## Collecting Our Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy.stats import pearsonr, spearmanr

All Datasets Used

In [ ]:
player_play = pd.read_csv("data/player_play.csv")
game_tracking_2023 = pd.read_csv("data/game_tracking_2023.csv")
game_tracking_2024 = pd.read_csv("data/game_tracking_2024.csv")
game_tracking_2025 = pd.read_csv("data/game_tracking_2025.csv")
combine_results = pd.read_csv("data/combine_results.csv")
combine_tracking = pd.read_csv("data/combine_tracking.csv")
players = pd.read_csv("data/players.csv")
career_success = pd.read_csv("data/player_career_successes.csv")

Filtering combine data for offensive-linemen

In [ ]:
oline = combine_results.loc[
    (combine_results['combine_position'] == 'OL'),
    ['nfl_id', 'combine_weight']
]

oline['weight'] = oline['combine_weight'] / 2.20462 # Convert to kg

Filtering player_play into passing and rushing plays based on if a play has a pass result or not.

In [ ]:
passing = player_play.loc[
    (player_play['lined_up_position'] == 'T') |
    (player_play['lined_up_position'] == 'G') |
    (player_play['lined_up_position'] == 'C'),
    ['game_id', 'play_id', 'nfl_id', 'lined_up_position', 'pass_result']
]

rushing = passing[passing['pass_result'].isna()] 

passing = passing[passing['pass_result'].notna()]

passing = passing.drop_duplicates(
    subset=['game_id', 'play_id', 'nfl_id']
) # Make sure no duplicate values

rushing = rushing.drop_duplicates(
    subset=['game_id', 'play_id', 'nfl_id']
) # Make sure no duplicate values

Combining all game tracking datasets, then merging them with our oline dataset to ensure only offensive linemen are selected.

In [ ]:
game_tracking = pd.concat([game_tracking_2023, game_tracking_2024, game_tracking_2025], ignore_index=True)

In [ ]:
oline_tracking = game_tracking.merge(
    oline[['nfl_id', 'weight']],
    on='nfl_id',
    how='inner'
)